# Power Query Data Transformation

Este notebook implementa a lógica do Power Query em Python, carregando e processando arquivos Excel de kit de colaboradores e equipamentos.

**Funcionalidades:**
- Carrega 3 arquivos Excel (Kit Promovido, Kit Novo, Solicitar Equipamento)
- Aplica transformações de dados
- Combina os dados
- Filtra por localidade
- Exporta resultado em Excel

## 1. Import Required Libraries

In [ ]:
import sys
from pathlib import Path

# Adiciona diretório raiz ao path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import numpy as np
from datetime import datetime
from core.data_transformer import transform_power_query, export_to_excel

print("✓ Bibliotecas carregadas com sucesso")

## 2. Configure File Paths

In [ ]:
# Configure os caminhos dos arquivos Excel
# IMPORTANTE: Atualize estes caminhos para a localização correta dos arquivos

kit_promovido_path = r"C:\Users\thallys.fernandes\Documents\Teste etiqueta\kit.colaborador.promovido.xlsx"
kit_novo_path = r"C:\Users\thallys.fernandes\Documents\Teste etiqueta\kit.novo.colaborador.xlsx"
solicitar_equipamento_path = r"C:\Users\thallys.fernandes\Documents\Teste etiqueta\Solicitar.equipamento.xlsx"

# Pasta de saída
output_folder = Path("output")
output_folder.mkdir(exist_ok=True)
output_path = output_folder / "dados_transformados.xlsx"

# Verifica se os arquivos existem
files_check = {
    "Kit Promovido": Path(kit_promovido_path).exists(),
    "Kit Novo": Path(kit_novo_path).exists(),
    "Solicitar Equipamento": Path(solicitar_equipamento_path).exists(),
}

for file_name, exists in files_check.items():
    status = "✓" if exists else "✗"
    print(f"{status} {file_name}: {'Encontrado' if exists else 'NÃO ENCONTRADO'}")

## 3. Execute Power Query Transformation

In [ ]:
# Executa a transformação
try:
    df = transform_power_query(
        kit_promovido_path,
        kit_novo_path,
        solicitar_equipamento_path,
    )
    print(f"\n✓ Transformação concluída com sucesso!")
    print(f"Total de registros: {len(df)}")
except Exception as e:
    print(f"✗ Erro ao transformar dados: {e}")
    raise

## 4. View Data Summary

In [ ]:
# Exibe informações do DataFrame
print("\n" + "="*80)
print("RESUMO DOS DADOS")
print("="*80)
print(f"\nColunas ({len(df.columns)}):")
print(f"  {', '.join(df.columns)}")

print(f"\nDimensões: {df.shape[0]} linhas × {df.shape[1]} colunas")

print(f"\nTipos de dados:")
for col, dtype in df.dtypes.items():
    print(f"  {col}: {dtype}")

## 5. Display First Records

In [ ]:
# Exibe as primeiras linhas
print(f"\nPrimeiros 5 registros:\n")
df.head(5)

## 6. Display Key Columns

In [ ]:
# Exibe apenas as colunas principais
key_cols = ["Chamado", "Nome Completo", "Centro de Custo", "Destino", "Equipamentos", "Data de retirada", "Origem"]
available_cols = [c for c in key_cols if c in df.columns]

print(f"\nColunas principais:\n")
df[available_cols].head(10)

## 7. Data Statistics

In [ ]:
# Estatísticas dos dados
print("\nESTATÍSTICAS")
print("="*80)

if "Origem" in df.columns:
    print(f"\nRegistros por Origem:")
    print(df["Origem"].value_counts())

if "Destino" in df.columns:
    print(f"\nRegistros por Destino:")
    print(df["Destino"].value_counts().head(10))

if "Centro de Custo" in df.columns:
    print(f"\nRegistros por Centro de Custo:")
    print(df["Centro de Custo"].value_counts().head(10))

# Valores nulos
print(f"\nValores nulos por coluna:")
null_counts = df.isna().sum()
null_cols = null_counts[null_counts > 0]
if len(null_cols) > 0:
    print(null_cols)
else:
    print("  Nenhum valor nulo encontrado")

## 8. Export to Excel

In [ ]:
# Exporta os dados para Excel
try:
    export_to_excel(df, str(output_path))
    print(f"\n✓ Arquivo exportado com sucesso!")
    print(f"Localização: {output_path.absolute()}")
except Exception as e:
    print(f"✗ Erro ao exportar: {e}")
    raise

## 9. Search Equipment Data

In [ ]:
# Busca específica em equipamentos
search_term = "notebook"  # Mude para o que quiser buscar

if "Equipamentos" in df.columns:
    filtered = df[df["Equipamentos"].str.contains(search_term, case=False, na=False)]
    print(f"\nRegistros com '{search_term}' em Equipamentos:")
    print(f"Total: {len(filtered)}")
    
    if len(filtered) > 0:
        print(f"\nPrimeiros 5 resultados:")
        filtered[["Chamado", "Nome Completo", "Equipamentos"]].head(5)

## 10. Integration with GlassPrinter

In [ ]:
# Exemplo de integração com o GlassPrinter
# Este código mostra como usar os dados transformados no sistema de etiquetas

# Formatar dados para etiquetas
etiquetas_data = df[[
    "Chamado",
    "Nome Completo",
    "Centro de Custo",
    "Destino",
    "Equipamentos",
    "Data de retirada"
]].copy()

print(f"\nDados formatados para etiquetas GlassPrinter:")
print(f"Total de linhas: {len(etiquetas_data)}")
print(f"\nPreview:")
etiquetas_data.head()